In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# 获取数据
path = Path('..') / 'data' / 'raw' / 'gapminder.tsv'
gd = pd.read_csv(path, sep='\t')
print(gd.shape)

In [ ]:
# 列名统一转小写，用下划线连接
gd = gd.rename(
    columns={
        'lifeExp':'life_exp',
        'gdpPercap':'gdp_percap'
    }
)

# 类型转换
gd = gd.astype({
    'year':int,
    'pop':int,
    'life_exp':float,
    'gdp_percap':float,
})

# 派生新列
gd['gdp_total'] = gd['gdp_percap'] * gd['pop']

# 逻辑一致性检查
gd = gd.query('life_exp > 0 and gdp_percap >0 and 1952 <= year <=2007')

# 缺失值与重复值
print(gd.isnull().sum()) # 无缺失值
print(gd.duplicated().sum()) # 无缺失值
gd = gd.drop_duplicates() 

# 保存
path = Path('..') / 'data' / 'clean' / 'gapminder_clean.csv'
gd.to_csv(path, index=False)
print(gd.shape)

In [ ]:
import plotly.express as px

# 确保 assets 文件夹存在
assets_dir = Path('..') / 'assets'
assets_dir.mkdir(exist_ok=True)

# 读取清洗后的数据
path = Path('..') / 'data' / 'clean' / 'gapminder_clean.csv'
gd = pd.read_csv(path)

# ============================================================
# 1. 折线图：各大洲预期寿命趋势
# ============================================================
df_trend = gd.groupby(['year', 'continent'], as_index=False)['life_exp'].mean()

fig1 = px.line(
    df_trend, x='year', y='life_exp', color='continent',
    title='各大洲预期寿命趋势', labels={'life_exp': '预期寿命', 'year': '年份'}
)
fig1.write_html(assets_dir / 'life_exp_line.html')
fig1.show()

# ============================================================
# 2. 柱状图：1952年 Top 10 国家（人均 GDP）
# ============================================================
target_year = 1952
df_1952 = gd[gd['year'] == target_year].nlargest(10, 'gdp_percap')

fig2 = px.bar(
    df_1952, x='gdp_percap', y='country', orientation='h',
    title=f'{target_year}年人均GDP Top 10 国家',
    labels={'gdp_percap': '人均GDP', 'country': '国家'},
    color='gdp_percap', color_continuous_scale='Blues'
)
fig2.update_layout(yaxis={'categoryorder':'total ascending'})
fig2.write_html(assets_dir / 'top10_bar.html')
fig2.show()

# ============================================================
# 3. 散点图（气泡图）：人均GDP vs 预期寿命，气泡=人口
# ============================================================
df_1952_scatter = gd[gd['year'] == target_year]

fig3 = px.scatter(
    df_1952_scatter, x='gdp_percap', y='life_exp', size='pop', color='continent',
    hover_name='country', log_x=True, # 对数轴处理GDP跨度
    title=f'{target_year}年人均GDP vs 预期寿命（气泡=人口）',
    labels={'gdp_percap': '人均GDP（对数轴）', 'life_exp': '预期寿命', 'continent': '大洲'}
)
fig3.write_html(assets_dir / 'gdp_vs_life_scatter.html')
fig3.show()

# ============================================================
# 4. 地图：2007年全球预期寿命分布
# ============================================================
# 注：Plotly 原生不支持 Timeline，直接画某一年即可，年份筛选留给 M3 的 Dash Slider 实现
df_2007 = gd[gd['year'] == 2007]

fig4 = px.choropleth(
    df_2007, locations='country', locationmode='country names',
    color='life_exp', hover_name='country',
    color_continuous_scale='Viridis',
    title='2007年全球预期寿命分布地图',
    labels={'life_exp': '预期寿命'}
)
fig4.write_html(assets_dir / 'life_exp_map.html')
fig4.show()